<a href="https://colab.research.google.com/github/LCaravaggio/FelicidadDesigualdad/blob/main/MNV2_Entrenada_contra_Gini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import userdata
import json

!mkdir ~/.kaggle
!touch ~/.kaggle/kaggle.json

api_token = {
    'username': userdata.get('KAGGLE_USER'),
    'key': userdata.get('KAGGLE_KEY')}
with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(api_token, file)

!chmod 600 ~/.kaggle/kaggle.json

import kagglehub
path4 = kagglehub.dataset_download("leonardocaravaggio/ge-images4")
path5 = kagglehub.dataset_download("leonardocaravaggio/ge-images5")

mkdir: cannot create directory ‘/root/.kaggle’: File exists


100%|██████████| 2.89G/2.89G [00:46<00:00, 66.8MB/s]

Extracting files...


100%|██████████| 895M/895M [00:12<00:00, 75.8MB/s]

Extracting files...


In [67]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from scipy.stats import pearsonr
from tqdm import tqdm


# ==== PARÁMETROS ====
IMG_TYPE = "10K"  # Puede ser "1K", "5K", "10K", "15K"
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==== CARGA ====
df_eph = pd.read_csv("Gini_EPH.csv")
df_ocde = pd.read_csv(path4 + "/Gini con latlon.csv")

# ==== NORMALIZAR COLUMNAS ====
# Renombrar columnas para que coincidan
df_eph = df_eph.rename(columns={"Nombre_Aglomerado": "Ciudad", "Gini_Hogares": "Gini"})
df_ocde["Gini"] = df_ocde["Gini"].str.replace(',', '.', regex=False).astype(float)

# ==== CONCATENAR ====
df_merged = pd.concat([df_eph[["Ciudad", "Gini"]], df_ocde[["Ciudad", "Gini"]]], ignore_index=True)


# ==== TRANSFORMACIONES ====
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def sanear_nombre_ciudad(nombre):
    return nombre.replace("/", ".").replace(":", "_").replace("'", "!")

# ==== DATASET PERSONALIZADO ====
import os
from PIL import Image

class GiniDataset(torch.utils.data.Dataset):
    def __init__(self, df, path1, path2, transform=None):
        self.df = df
        self.path1 = path1
        self.path2 = path2
        self.transform = transform
        self.sufijos = [" - 1K.png", " - 5K.png", " - 10K.png", " - 15K.png"]  # todos los sufijos que usás

    def sanear_nombre_ciudad(self, nombre):
        # Reemplazá acá caracteres conflictivos que tengas, y sacá backslashes si hay
        nombre = nombre.replace("/", ".").replace(":", "_").replace("'", "!")
        nombre = nombre.replace("\\", "")  # Sacar backslashes que aparezcan
        nombre = nombre.strip()
        return nombre

    def buscar_imagen(self, nombre_archivo):
        # Buscar imagen en path1 y path2 con cualquiera de los sufijos
        for base_path in [self.path1, self.path2]:
            for sufijo in self.sufijos:
                ruta = os.path.join(base_path, f"{nombre_archivo}{sufijo}")
                if os.path.exists(ruta):
                    return ruta
        return None

    def __getitem__(self, idx):
        nombre_ciudad = self.df.iloc[idx]["Ciudad"]
        nombre_archivo = self.sanear_nombre_ciudad(nombre_ciudad)

        img_path = self.buscar_imagen(nombre_archivo)
        if img_path is None:
            print(f"⚠️ No se encontró la imagen para {nombre_ciudad} en {self.path1} ni en {self.path2}")
            image = Image.new("RGB", (224, 224), (0, 0, 0))  # imagen negra placeholder
        else:
            image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(self.df.iloc[idx]["Gini"], dtype=torch.float32)
        return image, label

    def __len__(self):
        return len(self.df)


# ==== CARGA DE DATOS ====
train_df, val_df = train_test_split(df_merged, test_size=0.2, random_state=42)
train_dataset = GiniDataset(train_df, path4, path5, transform)
val_dataset = GiniDataset(val_df, path4, path5, transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# ==== MODELO ====
mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
mobilenet_features = nn.Sequential(*list(mobilenet.features))  # todos los bloques
for param in mobilenet_features[15:].parameters():
    param.requires_grad = False

model = nn.Sequential(
    mobilenet_features,                     # output: (batch_size, 1280, H, W)
    nn.AdaptiveAvgPool2d((1,1)),            # output: (batch_size, 1280, 1, 1)
    nn.Flatten(),                           # output: (batch_size, 1280)
    nn.Linear(1280, 64),                    # <-- cambio aquí: 1280 input features
    nn.ReLU(),
    nn.Linear(64, 1)
).to(DEVICE)


# ==== OPTIMIZADOR Y PÉRDIDA ====
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ==== ENTRENAMIENTO ====
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    for inputs, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE).unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Train Loss: {running_loss/len(train_loader):.4f}")

# ==== VALIDACIÓN ====
model.eval()
preds_val, trues_val = [], []
with torch.no_grad():
    for inputs, targets in val_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs).cpu().numpy().flatten()
        preds_val.extend(outputs)
        trues_val.extend(targets.numpy())

r_val, p_val = pearsonr(preds_val, trues_val)

# Evaluar también en entrenamiento para referencia
preds_train, trues_train = [], []
with torch.no_grad():
    for inputs, targets in train_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs).cpu().numpy().flatten()
        preds_train.extend(outputs)
        trues_train.extend(targets.numpy())

r_train, p_train = pearsonr(preds_train, trues_train)

print(f"\n📊 Pearson Train: {r_train:.3f} | p-value Train: {p_train:.5f}")
print(f"📊 Pearson Val: {r_val:.3f} | p-value Val: {p_val:.5f}")

Epoch 1/10: 100%|██████████| 8/8 [02:45<00:00, 20.69s/it]


Train Loss: 0.0186


Epoch 2/10: 100%|██████████| 8/8 [02:31<00:00, 18.92s/it]


Train Loss: 0.0056


Epoch 3/10: 100%|██████████| 8/8 [02:39<00:00, 19.99s/it]


...

Train Loss: 0.0008


Epoch 9/10: 100%|██████████| 8/8 [02:21<00:00, 17.64s/it]


Train Loss: 0.0006


Epoch 10/10: 100%|██████████| 8/8 [02:24<00:00, 18.07s/it]


Train Loss: 0.0006

📊 Pearson Train: 0.955 | p-value Train: 0.00000
📊 Pearson Val: 0.418 | p-value Val: 0.02149
